# DIY nanoGPT Training (Andrej-Style)

This notebook incorporates Andrej Karpathy's training optimizations:
- **AdamW** with selective weight decay
- **Cosine LR schedule** with linear warmup
- **Gradient clipping** (1.0)
- **Mixed precision** (bfloat16/float16)
- **Gradient accumulation** for larger effective batch size

Uses the DIY_nanoGPT model with your custom dataset.

## 1. Environment Setup & Imports

In [ ]:
import os
import math
import time
import json
import glob
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# Check environment
def is_colab() -> bool:
    try:
        import google.colab
        return True
    except Exception:
        return False

project_name = "nanoGPT"

if is_colab():
    from google.colab import drive
    drive.mount('COLAB_DRIVE')
    matches = glob.glob(f"COLAB_DRIVE/MyDrive/**/{project_name}/", recursive=True)
    print(matches)
    if len(matches) == 1:
        proj_dir = matches[0]
else:
    from root_path import get_root_path
    proj_dir = get_root_path()
    print("Running locally")

# Load config
with open(os.path.join(proj_dir, "config.json"), "r") as f:
    config = json.load(f)

import sys
sys.path.append(proj_dir)

from utils import torch_ckpt
from model import model

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"bfloat16 supported: {torch.cuda.is_bf16_supported()}")

## 2. Training Configuration (Andrej-style)

In [ ]:
# ============= Training Hyperparameters (Andrej-style) =============

# I/O
out_dir = os.path.join(proj_dir, "ckpt")
eval_interval = 100
log_interval = 10
eval_iters = 20
always_save_checkpoint = False  # Only save on best val loss

# Data
batch_size = 4  # Micro-batch size (keep same as DIY_nanoGPT)
gradient_accumulation_steps = 8  # Effective batch = 4 * 8 = 32

# Model (from existing config)
token_len = config['deep_learning_settings']['model_config']['token_len']  # 1024

# AdamW Optimizer (Andrej-style)
learning_rate = 6e-4  # Peak learning rate
min_lr = 6e-5  # Minimum learning rate (~10% of peak)
weight_decay = 0.1  # Only applied to 2D+ params
beta1 = 0.9
beta2 = 0.95  # Lower than default 0.999
grad_clip = 1.0  # Gradient clipping

# Learning Rate Schedule
warmup_iters = 100  # Linear warmup steps
lr_decay_iters = 5000  # When to stop decay (then constant at min_lr)
decay_lr = True

# Training
max_iters = 10000
patience = 20  # Early stopping patience

# System
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
compile_model = False  # Set True for PyTorch 2.0+ (slower startup, faster training)

# Seed
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

print("="*60)
print("Training Configuration (Andrej-style)")
print("="*60)
print(f"Device: {device}")
print(f"Data type: {dtype}")
print(f"Micro-batch size: {batch_size}")
print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
print(f"Effective batch size: {batch_size * gradient_accumulation_steps}")
print(f"Context length: {token_len}")
print(f"Tokens per iteration: {batch_size * gradient_accumulation_steps * token_len:,}")
print(f"Peak LR: {learning_rate}, Min LR: {min_lr}")
print(f"Warmup iters: {warmup_iters}, Decay iters: {lr_decay_iters}")
print(f"Weight decay: {weight_decay}")
print(f"Gradient clipping: {grad_clip}")
print("="*60)

## 3. Mixed Precision & Performance Setup

In [ ]:
# Enable TF32 for faster matmul (Ampere+ GPUs)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Setup mixed precision context
device_type = 'cuda' if 'cuda' in device else 'cpu'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

# Autocast context for forward pass
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

# GradScaler for float16 (not needed for bfloat16)
scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

print(f"Using {'mixed precision' if device_type == 'cuda' else 'float32'} training")
print(f"Autocast dtype: {ptdtype}")
print(f"GradScaler enabled: {dtype == 'float16'}")

## 4. Data Loader

In [ ]:
from data.data_loader import TokenDataLoader

data_loader = TokenDataLoader(
    data_train_path=os.path.join(proj_dir, "data", "tokenized", "concatenated", "train_all.bin"),
    data_val_path=os.path.join(proj_dir, "data", "tokenized", "concatenated", "val_all_10_to_11.bin"),
    token_len=token_len,
    batch_size=batch_size
)

print(f"Training data loaded from: data/tokenized/concatenated/train_all.bin")
print(f"Validation data loaded from: data/tokenized/concatenated/val_all_10_to_11.bin")

## 5. Model Initialization

In [ ]:
# Remove ff_dim if present (not used in model)
model_config = config['deep_learning_settings']['model_config'].copy()
model_config.pop('ff_dim', None)

# Create model
gpt_model = model.GPT(**model_config)
gpt_model.to(device)

# Optionally compile model (PyTorch 2.0+)
if compile_model:
    print("Compiling model... (this takes ~1 minute)")
    gpt_model = torch.compile(gpt_model)

# Count parameters
n_params = sum(p.numel() for p in gpt_model.parameters())
print(f"Model parameters: {n_params:,} ({n_params/1e6:.2f}M)")

## 6. Optimizer Setup (Andrej-style AdamW)

In [ ]:
def configure_optimizers(model, weight_decay, learning_rate, betas, device_type):
    """
    Configure AdamW optimizer with parameter groups.
    Weight decay is only applied to 2D+ parameters (weights, embeddings).
    1D parameters (biases, LayerNorm) do not get weight decay.
    """
    # Separate parameters into decay and no-decay groups
    param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
    
    decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
    
    optim_groups = [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0}
    ]
    
    num_decay_params = sum(p.numel() for p in decay_params)
    num_nodecay_params = sum(p.numel() for p in nodecay_params)
    print(f"Decay params: {num_decay_params:,} in {len(decay_params)} tensors")
    print(f"No-decay params: {num_nodecay_params:,} in {len(nodecay_params)} tensors")
    
    # Use fused AdamW if available (faster on CUDA)
    import inspect
    fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
    use_fused = fused_available and device_type == 'cuda'
    extra_args = dict(fused=True) if use_fused else dict()
    
    optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
    print(f"Using fused AdamW: {use_fused}")
    
    return optimizer

# Create optimizer
optimizer = configure_optimizers(
    gpt_model, 
    weight_decay=weight_decay, 
    learning_rate=learning_rate, 
    betas=(beta1, beta2),
    device_type=device_type
)

## 7. Learning Rate Scheduler (Cosine with Warmup)

In [ ]:
def get_lr(it):
    """
    Cosine learning rate schedule with linear warmup.
    
    1. Linear warmup from 0 to learning_rate over warmup_iters
    2. Cosine decay from learning_rate to min_lr over lr_decay_iters
    3. Constant at min_lr after lr_decay_iters
    """
    # 1) Linear warmup
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    
    # 2) After decay period, return minimum LR
    if it > lr_decay_iters:
        return min_lr
    
    # 3) Cosine decay between warmup and decay iters
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))  # Goes from 1 to 0
    return min_lr + coeff * (learning_rate - min_lr)

# Visualize the learning rate schedule
import matplotlib.pyplot as plt

lrs = [get_lr(i) for i in range(max_iters)]
plt.figure(figsize=(10, 4))
plt.plot(lrs)
plt.xlabel('Iteration')
plt.ylabel('Learning Rate')
plt.title('Cosine LR Schedule with Linear Warmup')
plt.axvline(x=warmup_iters, color='r', linestyle='--', label=f'Warmup end ({warmup_iters})')
plt.axvline(x=lr_decay_iters, color='g', linestyle='--', label=f'Decay end ({lr_decay_iters})')
plt.legend()
plt.grid(True)
plt.show()

## 8. Loss Estimation Function

In [ ]:
@torch.no_grad()
def estimate_loss():
    """
    Estimate train and validation loss over multiple batches.
    Returns averaged losses.
    """
    out = {}
    gpt_model.eval()
    
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            if split == 'train':
                x, y = data_loader.get_train_batch()
            else:
                x, y = data_loader.get_val_batch()
            
            x, y = x.to(device), y.to(device)
            
            with ctx:
                pred = gpt_model(x)
                B, T, C = pred.size()
                loss = F.cross_entropy(pred.view(B*T, C), y.view(B*T))
            
            losses[k] = loss.item()
        
        out[split] = losses.mean()
    
    gpt_model.train()
    return out

## 9. Checkpoint Manager Setup

In [ ]:
# Setup checkpoint paths
file_path = os.path.join(proj_dir, "Trainer", "Trainer_DIYModel_AndrejStyle.ipynb")

config['path_settings']['file_path'] = file_path
config['git_settings']['strict_git'] = False

# Initialize checkpoint manager
ckpt_manager = torch_ckpt.ckpt_manager(**config)

# Update with model and optimizer
ckpt_manager.deep_learning_settings.model = gpt_model
ckpt_manager.deep_learning_settings.optimizer = optimizer
ckpt_manager.deep_learning_settings.scheduler = None
ckpt_manager.deep_learning_settings.grad_scaler = scaler

print(f"Checkpoint manager initialized")
print(f"Output directory: {ckpt_manager.session_dir if hasattr(ckpt_manager, 'session_dir') else out_dir}")

## 10. Training Loop (Andrej-style)

In [ ]:
print("="*60)
print("Starting Training (Andrej-style)")
print("="*60)

# Training state
train_loss_history = {}
val_loss_history = {}
best_val_loss = float('inf')
patience_counter = 0

# Timing
t0 = time.time()
local_iter_num = 0

# Pre-fetch first batch
x, y = data_loader.get_train_batch()
x, y = x.to(device), y.to(device)

for iter_num in range(max_iters):
    # ============= Learning Rate Update =============
    if decay_lr:
        lr = get_lr(iter_num)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
    else:
        lr = learning_rate
    
    # ============= Evaluation =============
    if iter_num % eval_interval == 0:
        losses = estimate_loss()
        train_loss_history[iter_num] = {'loss': losses['train'], 'batch_size': batch_size}
        val_loss_history[iter_num] = {'loss': losses['val'], 'batch_size': batch_size}
        
        print(f"Step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr {lr:.6f}")
        
        # Check for improvement
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            patience_counter = 0
            
            if iter_num > 0:
                print(f"  -> New best val loss! Saving checkpoint...")
                ckpt_manager.save_ckpt(
                    step=iter_num,
                    train_loss_history=train_loss_history,
                    val_loss_history=val_loss_history,
                    best_val_loss=best_val_loss,
                    patience_counter=patience_counter
                )
        else:
            patience_counter += 1
            print(f"  -> No improvement. Patience: {patience_counter}/{patience}")
        
        # Early stopping check
        if patience_counter >= patience:
            print(f"\nEarly stopping at iter {iter_num}: no improvement for {patience} evaluations")
            break
    
    # ============= Training with Gradient Accumulation =============
    optimizer.zero_grad(set_to_none=True)
    
    for micro_step in range(gradient_accumulation_steps):
        # Forward pass with mixed precision
        with ctx:
            pred = gpt_model(x)
            B, T, C = pred.size()
            loss = F.cross_entropy(pred.view(B*T, C), y.view(B*T))
            loss = loss / gradient_accumulation_steps  # Scale for accumulation
        
        # Pre-fetch next batch during backward (async)
        x, y = data_loader.get_train_batch()
        x, y = x.to(device), y.to(device)
        
        # Backward pass with scaler
        scaler.scale(loss).backward()
    
    # ============= Gradient Clipping =============
    if grad_clip != 0.0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(gpt_model.parameters(), grad_clip)
    
    # ============= Optimizer Step =============
    scaler.step(optimizer)
    scaler.update()
    
    # ============= Logging =============
    if iter_num % log_interval == 0:
        t1 = time.time()
        dt = t1 - t0
        t0 = t1
        
        # Estimate loss value (unscaled)
        lossf = loss.item() * gradient_accumulation_steps
        
        if iter_num > 0:
            tokens_per_sec = (batch_size * gradient_accumulation_steps * token_len * log_interval) / dt
            print(f"Iter {iter_num}: loss {lossf:.4f}, lr {lr:.6f}, {dt*1000/log_interval:.1f}ms/iter, {tokens_per_sec:.0f} tok/s")
    
    local_iter_num += 1

# ============= Final Checkpoint =============
print(f"\nSaving final checkpoint at step {iter_num}...")
ckpt_manager.save_ckpt(
    step=iter_num,
    train_loss_history=train_loss_history,
    val_loss_history=val_loss_history,
    best_val_loss=best_val_loss,
    patience_counter=patience_counter
)

print("\n" + "="*60)
print("Training Complete!")
print(f"Best validation loss: {best_val_loss:.4f}")
print("="*60)

## 11. Training Loss Visualization

In [ ]:
import matplotlib.pyplot as plt

# Extract losses
train_iters = sorted(train_loss_history.keys())
train_losses = [train_loss_history[i]['loss'] for i in train_iters]
val_losses = [val_loss_history[i]['loss'] for i in train_iters]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_iters, train_losses, label='Train Loss', alpha=0.8)
plt.plot(train_iters, val_losses, label='Val Loss', alpha=0.8)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(train_iters, train_losses, label='Train Loss', alpha=0.8)
plt.plot(train_iters, val_losses, label='Val Loss', alpha=0.8)
plt.xlabel('Iteration')
plt.ylabel('Loss (log scale)')
plt.yscale('log')
plt.title('Training Progress (Log Scale)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"\nFinal train loss: {train_losses[-1]:.4f}")
print(f"Final val loss: {val_losses[-1]:.4f}")
print(f"Best val loss: {best_val_loss:.4f}")

## 12. Text Generation Test

In [ ]:
import tiktoken

# Load tokenizer
enc = tiktoken.get_encoding("gpt2")

# Generate text
gpt_model.eval()

# Start token
start_text = "The meaning of life is"
start_ids = enc.encode(start_text)
x = torch.tensor(start_ids, dtype=torch.long, device=device).unsqueeze(0)

print(f"Prompt: {start_text}")
print("="*50)

# Generate
with torch.no_grad():
    with ctx:
        for _ in range(200):  # Generate 200 tokens
            # Crop context if too long
            x_cond = x if x.size(1) <= token_len else x[:, -token_len:]
            
            # Forward
            logits = gpt_model(x_cond)
            logits = logits[:, -1, :]  # Last position
            
            # Temperature and sampling
            temperature = 0.8
            logits = logits / temperature
            
            # Top-k sampling
            top_k = 40
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')
            
            # Sample
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            
            # Append
            x = torch.cat((x, next_id), dim=1)

# Decode
generated_text = enc.decode(x[0].tolist())
print(generated_text)

## Summary

This notebook implements Andrej Karpathy's training optimizations:

1. **AdamW Optimizer** with selective weight decay (0.1 for weights, 0 for biases/LayerNorm)
2. **Cosine LR Schedule** with linear warmup (6e-4 -> 6e-5)
3. **Gradient Clipping** at 1.0 global norm
4. **Mixed Precision** (bfloat16 or float16 with GradScaler)
5. **Gradient Accumulation** for larger effective batch size
6. **Fused AdamW** (when available) for faster optimization
7. **TF32** enabled for faster matrix operations

Compare the loss curves with your original Trainer.ipynb to see the impact of these optimizations!